In [61]:
from openai import OpenAI
import requests
from pydantic_ai import Agent 
import os

from dotenv import load_dotenv

In [62]:

def get_page(title: str) -> str:
    """
    Retrieve the raw content of a Wikipedia page.

    Args:
        title (str): The exact title of the Wikipedia page.

    Returns:
        str: The page content as a string.
    """

    # Step 1: Define the Wikipedia Get Page API URL.
    url = "https://en.wikipedia.org/w/index.php"

    # Step 2: Specify the page title and request raw content.
    params = {
        "title": title,
        "action": "raw"
    }

    # Step 3: Define the required User-Agent header.
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }

    # Step 4: Send an HTTP GET request to Wikipedia.
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=15
    )

    # Step 5: Check whether the request succeeded.
    response.raise_for_status()

    # Step 6: Return the raw page content as a string.
    return response.text


In [63]:
 
def search_wikipedia(query: str) -> list[dict]:
    """
    Search Wikipedia for articles matching a query string.

    Args:
        query (str): The search term to look up on Wikipedia.

    Returns:
        list[dict]: A list of Wikipedia search result objects.
    """

    # Wikipedia Search API endpoint.
    url = "https://en.wikipedia.org/w/api.php"

    # API parameters.
    # requests will automatically encode spaces in the query.
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query
    }

    # Wikipedia requires a User-Agent header.
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }

    # Send a GET request to Wikipedia.
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=15
    )

    # Raise an exception if the HTTP request failed.
    response.raise_for_status()

    # Convert the JSON response into a Python dictionary.
    data = response.json()

    # Extract the search results from query.search.
    search_results = data["query"]["search"]

    # Return the list of matching articles.
    return search_results


In [64]:

# Search Wikipedia for the word "capybara".
# Task1: Search Wikipedia for the word "capybara" and print the number of results and the first result.
search_results = search_wikipedia("capybara")

# Print the number of results returned by the API.
print("Number of search results:", len(search_results))

# Display the first search result.
print(search_results[0])


Number of search results: 10
{'ns': 0, 'title': 'Capybara', 'pageid': 6776, 'size': 44907, 'wordcount': 4312, 'snippet': 'The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South', 'timestamp': '2026-10-21T15:25:43Z'}


In [65]:

# Iterate through every search result.
for result in search_results:

    # Extract the title field from the result.
    title = result["title"]

    # Print the article title.
    print(title)


Capybara
Lesser capybara
Capybara Games
Capybara (disambiguation)
Capybara Party
Capybara (software)
Hydrochoerus
Caviidae
Flow (2024 film)
List of largest rodents


In [66]:


# Question 2: Search Wikipedia for "capybara" again.

# Step 1: Call our search_wikipedia function again.
search_results = search_wikipedia("capybara")

# Count all results.
print("Total returned:", len(search_results))

# Step 2: Create an empty list to store matching titles.
capybara_titles = []

# Step 3: Examine each result in the query.search array.
for result in search_results:

    # Extract the title of the Wikipedia article.
    title = result["title"]

    # Convert the title to lowercase for case-insensitive matching.
    if "capybara" in title.lower():

        # Add the matching title to our list.
        capybara_titles.append(title)

# Step 4: Print all matching titles.
print("Titles containing 'capybara':")

for title in capybara_titles:
    print(title)

# Step 5: Count the matching titles.
print("\nNumber of matching titles:", len(capybara_titles))



Total returned: 10
Titles containing 'capybara':
Capybara
Lesser capybara
Capybara (disambiguation)
Capybara Games
Capybara Party
Capybara (software)

Number of matching titles: 6


In [67]:

# Step 1: Call get_page to fetch the Capybara article.
# This sends a new HTTP request to Wikipedia.
# Task 3 Define the Get Page Tool
content = get_page("Capybara")

# Step 2: Check the returned data type.
print("Content type:", type(content))

# Step 3: Count characters in the article.
character_count = len(content)

# Step 4: Print the number of characters.
print("Number of characters:", character_count)

# Step 5: Display the first 500 characters.
print("\nFirst 500 characters:")
print(content[:500])


Content type: <class 'str'>
Number of characters: 44771

First 500 characters:
{{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=''Hydrochoerus hydrochaeris'' |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en}}</ref>
| fossil_range = {{Fossilrange|Late Pleistocene|Present}}
| image = Capybaracropped.jpg
| image_alt = A capybara 


In [68]:

# Define instructions explaining how the agent should work.

instructions = """
You are a Wikipedia research assistant.

Answer user questions using information from Wikipedia.

Available tools:
1. search_wikipedia: Search Wikipedia for relevant pages.
2. get_page: Retrieve the full content of a Wikipedia page.

When answering a question about a topic:
- First, search Wikipedia for relevant articles.
- Then, retrieve the most relevant page using get_page.
- Use the retrieved information to formulate your answer.

When the user provides a specific Wikipedia page URL:
- Extract the page title from the URL.
- Use get_page to retrieve that page directly.

Provide clear and concise answers.

Use only facts found in the retrieved Wikipedia content.
If you cannot find the answer, inform the user.
"""


In [69]:

# Load the environment variables from .env.
load_dotenv(override=True)

# Create a PydanticAI agent.

wikipedia_agent = Agent(

    # Name used to identify the agent.
    name="wikipedia",

    # OpenAI model used by the agent.
    model="openai:gpt-4o-mini",

    # Instructions controlling agent behaviour.
    instructions=instructions,

    # Register the two tools created in Questions 1 and 3.
    tools=[
        search_wikipedia,
        get_page
    ]
)

In [70]:

# Ask the Wikipedia agent a question.
# Task 4: Ask the Wikipedia agent a question and print the answer.

# Step 1: Define the question.
question = (
    "What is this page about? "
    "https://en.wikipedia.org/wiki/Capybara"
)

# Step 2: Run the Wikipedia agent.
agent_result = await wikipedia_agent.run(question)

# Step 3: Print the final answer.
print("AGENT ANSWER:")
print(agent_result.output)

# Step 4: Get the token usage information.
usage = agent_result.usage

# Step 5: Print token counts.
print("\nTOKEN USAGE:")
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

# Step 4: Verify that get_page was called.
print("\nTOOL CALLS:")

for message in agent_result.all_messages():
    for part in message.parts:
        if part.part_kind == "tool-call":
            print(part.tool_name, part.args)



AGENT ANSWER:
The Wikipedia page on the **capybara** (scientific name: *Hydrochoerus hydrochaeris*) provides detailed information about this animal, which is the largest living rodent. Here are some key points covered on the page:

- **Habitat and Distribution**: Capybaras are native to South America, found in various countries except Chile. They are semi-aquatic and typically inhabit savannas, wetlands, and areas near freshwater bodies.

- **Physical Characteristics**: They possess a heavy, barrel-shaped body, reddish-brown fur on top, and a lighter underside. Adult capybaras can weigh between 35 and 66 kg, and they have slightly webbed feet.

- **Social Behavior**: Capybaras are very social animals, usually living in groups of 10-20, but sometimes in groups of up to 100 individuals. They communicate using a variety of vocalizations.

- **Diet**: Being herbivores, they primarily consume grasses and aquatic plants. They are known to practice coprophagy, eating their own feces to maximi

In [71]:

# Step 1: Use the exact question from the homework.
question = "What are the main threats to capybara populations?"

# Step 2: Run the existing PydanticAI Wikipedia agent.
result_q6 = await wikipedia_agent.run(question)

# Step 3: Print the agent's final answer.
print("AGENT ANSWER:")
print(result_q6.output)

# Step 4: Display token usage.
print("\nTOKEN USAGE:")
print(result_q6.usage)


AGENT ANSWER:
Capybara populations face several threats, though they are currently listed as a species of "Least Concern" due to a stable population across much of their range in South America. Here are the main threats:

1. **Hunting**: Capybaras are hunted for their meat and pelts. In some regions, excessive hunting has led to a decline in local populations.

2. **Human Encroachment**: Habitat destruction due to agriculture, urbanization, and deforestation reduces their natural habitats. This may lead to conflicts with humans, who often view capybaras as competitors for resources.

3. **Predation**: Natural predators such as jaguars, caimans, and various birds of prey also threaten young capybaras, impacting their survival rates.

4. **Livestock Competition**: In some areas, capybaras are perceived as competition for grazing resources, which can lead to attempts to control or eliminate their populations.

5. **Disease**: Capybaras can be vectors for diseases, including Rocky Mountain

In [73]:

# Get all tool calls from the agent's message history.
tool_calls = [
    part
    for message in result_q6.all_messages()
    for part in message.parts
    if part.part_kind == "tool-call"
]

# Display tool calls in order.
print("TOOLS USED IN ORDER:")

for i, call in enumerate(tool_calls, start=1):
    print(f"{i}. {call.tool_name}({call.args})")

# Display the total number of tool calls.
print("\nTOTAL TOOL CALLS:", len(tool_calls))

# Display the final answer.
print("\nFINAL AGENT ANSWER:")
print(result_q6.output)


TOOLS USED IN ORDER:
1. search_wikipedia({"query":"Capybara threats"})
2. search_wikipedia({"query":"Capybara"})
3. get_page({"title":"Capybara"})

TOTAL TOOL CALLS: 3

FINAL AGENT ANSWER:
Capybara populations face several threats, though they are currently listed as a species of "Least Concern" due to a stable population across much of their range in South America. Here are the main threats:

1. **Hunting**: Capybaras are hunted for their meat and pelts. In some regions, excessive hunting has led to a decline in local populations.

2. **Human Encroachment**: Habitat destruction due to agriculture, urbanization, and deforestation reduces their natural habitats. This may lead to conflicts with humans, who often view capybaras as competitors for resources.

3. **Predation**: Natural predators such as jaguars, caimans, and various birds of prey also threaten young capybaras, impacting their survival rates.

4. **Livestock Competition**: In some areas, capybaras are perceived as competitio